## We need to build the provider setup for our LLM's so that whenever we ask a question we have a flexibility using which model we want the answer

In [1]:
from dataclasses import dataclass
from typing import Optional
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

print("GROQ:", bool(os.getenv("GROQ_API_KEY")))
print("OPENROUTER:", bool(os.getenv("OPENROUTER_API_KEY")))

@dataclass(frozen=True)
class Provider():
    """ One Provider to reliably route request across all inference providers"""
    name: str
    env_var: str
    is_free: bool
    base_url: Optional[str]
    model: Optional[str]

PROVIDERS = [
    Provider("Groq", "GROQ_API_KEY", True, "https://api.groq.com/openai/v1", "openai/gpt-oss-120b"),
    Provider("OpenRouter", "OPENROUTER_API_KEY", True, "https://openrouter.ai/api/v1", "openrouter/free"),
]

def select_provider():
    for provider in PROVIDERS:
        if(os.getenv(provider.env_var)):
            return provider;

    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider API Key is set add one of the {expected} to your environment variables")


# Now let's build a client
def build_client(provider: Provider) -> OpenAI:
    api_key = os.getenv(provider.env_var)
    if provider.base_url is None:
        return OpenAI(api_key=api_key)

    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url
    )
    

def have_any_key():
    return any(os.getenv(provider.env_var) for provider in PROVIDERS)


print("Found a provider key" if have_any_key() else "No Provider key found")


# Let's use call_provider() whose responsibility is to just call the provider and return the response
def call_provider(prompt: str, provider: Provider) -> str:
    client = build_client(provider)

    result = client.chat.completions.create(
        model=provider.model,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )

    return result.choices[0].message.content
    

GROQ: True
OPENROUTER: True
Found a provider key


In [2]:
def available_providers():
    return [provider for provider in PROVIDERS if (os.getenv(provider.env_var))]


def llm_reply(prompt: str) -> str:
    providers = available_providers()

    if not providers:
        expected = ", ".join(
            provider.env_var
            for provider in PROVIDERS
        )

        raise RuntimeError(
            f"No provider API key is set. "
            f"Add one of: {expected}"
        )

    last_error = None

    for provider in providers:

        print(f"Trying {provider.name}...")

        try:
            return call_provider(prompt, provider)

        except Exception as e:
            last_error = e

            status_code = getattr(e, "status_code", None)

            if status_code == 429 or os.getenv(provider.env_var) is None:
                print(
                    f"{provider.name} rate limit reached or there is no {provider.name} provider"
                    f"Trying next provider..."
                )
                continue

            # Don't fallback for every possible error.
            # Authentication/configuration/model errors
            # should be visible immediately.
            raise

    raise RuntimeError(
        "All configured LLM providers failed."
    ) from last_error


In [3]:
print(llm_reply("Who was the PM of India in 1980 ?"))

Trying OpenRouter...
Indira Gandhi was the Prime Minister of India in 1980.


In [32]:
print(llm_reply("Who was before him ?"))

Trying Groq...
I’m not sure which person you’re asking about. Could you let me know who “him” refers to? That way I can give you the right answer.
